# Adaptive Preprocessing Selection for Domain-Specific NLP Pipelines
## A Meta-Learning Approach

**Team Members:**
- **Ritesh Sharma** (USN: ENG25CSE012)
- **Longjam Nirish Singh** (USN: ENG25CSE031)

---
### Abstract & Research Overview
Text preprocessing is traditionally treated as a fixed chore—lowercasing tokens, removing stopwords, and stripping punctuation. However, when heterogeneous texts from biomedical literature, financial statements, and social media feeds are cleaned with the same aggressive recipe, critical domain signals are inadvertently eradicated.

This system provides **Adaptive NLP Preprocessing for Domain-Specific Text**—an intelligent meta-learning gateway that:
1. **Partitions** incoming text streams into semantic micro-clusters using MiniLM sentence embeddings + Spherical K-Means ($k=2-6$).
2. **Profiles** each micro-cluster with a 9-dimensional label-free statistical meta-vector (TTR, uppercase ratio, digit density, punctuation entropy, POS entropy, dependency depth, OOV rate, BPE fragmentation, and geometric dispersion).
3. **Routes** each micro-cluster to one of four specialized preprocessing policies using a calibrated LightGBM surrogate ranker.
4. **Executes** the chosen policy ($P_0$ Minimal, $P_1$ Aggressive Social, $P_2$ Negation Shielded, $P_3$ Biomedical Entity Preserving) to maximize downstream signal retention.

### 1. Install & Verify Dependencies

In [ ]:
# Install dependencies (if running in Google Colab)
!pip install -q lightgbm scikit-learn pandas numpy scipy streamlit

### 2. Preprocessing Policy Space ($P_0$ to $P_3$)
Four interpretable policies with a shared first step (URL stripping & emoji mapping):
- **P0 (Minimal / Technical):** Whitespace normalization only. Case, numbers, and symbols preserved.
- **P1 (Aggressive Social):** Lowercase, strip handles and punctuation, collapse whitespace.
- **P2 (Negation Shielded):** Lowercase; prefix tokens following negation markers with `NOT_` until punctuation.
- **P3 (Biomedical Entity Preserving):** Preserve case, hyphenated drug/disease compounds, units, and chemical delimiters.

In [ ]:
from src.policy_engine import DomainPolicyEngine, POLICY_NAMES, POLICY_DESCRIPTIONS

sample_text = "Targeting VEGFR2 with 14.5 mg/kg ramucirumab did NOT improve @patient_01! :("
print(f"Original text: {sample_text}\n")

for pid in range(4):
    cleaned = DomainPolicyEngine.apply_policy(sample_text, pid)
    print(f"[{POLICY_NAMES[pid]}]\n  --> {cleaned}\n")

### 3. Micro-Cluster Partitioning & 9-D Profiling
Incoming batches are embedded via MiniLM, normalized, clustered into micro-batches, and summarized with 9 statistical metrics without requiring any domain labels.

In [ ]:
from src.partitioner import MicroClusterPartitioner
from src.profiler import LocalMetaProfiler
import pandas as pd

partitioner = MicroClusterPartitioner()
profiler = LocalMetaProfiler()

sample_batch = [
    "Targeting of VEGFR2 signaling with ramucirumab successfully limited metastatic progression.",
    "The company reported EUR 12.4M in pre-tax losses due to unexpected restructuring costs.",
    "no way, this is absolutely unacceptable! never purchasing again, worst delivery service. 😡"
]

clusters = partitioner.partition(sample_batch, k=3)
cluster_profiles = []
for c_id, items in clusters.items():
    texts = [it[0] for it in items]
    embs = [it[1] for it in items]
    meta_vec = profiler.extract(texts, embs)
    row = {"Cluster": c_id, "Size": len(texts)}
    for fname, val in zip(profiler.FEATURE_NAMES, meta_vec):
        row[fname] = round(float(val), 4)
    cluster_profiles.append(row)

df_prof = pd.DataFrame(cluster_profiles)
display(df_prof)

### 4. Calibrated LightGBM Surrogate Policy Router
Trained on 400 calibrated empirical vectors (100 per policy) with Gaussian noise $\sigma = 0.01$ to eliminate policy collapse and provide high-confidence routing.

In [ ]:
from src.router import LightGBMSurrogateRouter

router = LightGBMSurrogateRouter()
print("LightGBM surrogate policy router loaded.")

# Test router prediction on extracted profile
for c_id, row in df_prof.iterrows():
    vec = row[profiler.FEATURE_NAMES].values.astype(float)
    p_id, probs, conf = router.predict(vec)
    print(f"Cluster {row['Cluster']} -> Policy: {POLICY_NAMES[p_id]} (Confidence: {conf*100:.1f}%)")

### 5. Full End-to-End Gateway Execution
Executes Partition -> Profile -> Align -> Route -> Clean pipeline.

In [ ]:
from src.gateway import AdaptivePreprocessingGateway

gateway = AdaptivePreprocessingGateway()
results = gateway.process_batch(sample_batch, k_clusters=3)

for r in results:
    print(f"[Cluster {r['assigned_cluster']}] Policy: {r['routing_decision']} ({r['confidence']})")
    print(f"  Input: {r['original_text']}")
    print(f"  Clean: {r['cleaned_text']}\n")

### 6. Experimental Evaluation Suite (Reproducing Project Report Findings)
- **Table 6.1:** Signal Retention Analysis
- **Table 6.2 & 6.3:** Gateway Routing on Mixed Stream
- **Table 6.4 & 6.5:** Downstream Representation Evaluation & Normalized Regret

In [ ]:
from src.evaluation import run_all

run_all()